In [1]:
import os
import subprocess
import shutil


# paths
INPUT_FOLDER  = r"E:\8th-Semester-FAST-NUCES-Peshawar\Final preparation vedio labeling\final codes\model\test codes"
OUTPUT_FOLDER = r"E:\8th-Semester-FAST-NUCES-Peshawar\Final preparation vedio labeling\final codes\model\test codes\standardized_vedio"

TARGET_FPS    = 25
TARGET_WIDTH  = 1080
TARGET_HEIGHT = 1920
FPS_TOLERANCE = 1.0

VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv')


def get_video_info(video_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,r_frame_rate",
        "-of", "default=noprint_wrappers=1",
        video_path
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        lines  = result.stdout.strip().split("\n")
        width  = int(lines[0].split("=")[1])
        height = int(lines[1].split("=")[1])
        num, den = lines[2].split("=")[1].split("/")
        return width, height, float(num) / float(den)
    except:
        return 0, 0, 0


def normalize_video(input_path, output_path):
    vf = (
        f"scale={TARGET_WIDTH}:{TARGET_HEIGHT}:force_original_aspect_ratio=decrease,"
        f"pad={TARGET_WIDTH}:{TARGET_HEIGHT}:(ow-iw)/2:(oh-ih)/2,"
        f"fps={TARGET_FPS}"
    )
    cmd = [
        "ffmpeg", "-i", input_path,
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "18",
        "-y", output_path
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        print("ffmpeg not found — install it and add to PATH")
        return False


class VideoNormalizer:

    def __init__(self, input_folder, output_folder):
        self.input_folder  = input_folder
        self.output_folder = output_folder
        self.videos        = []
        self.converted     = 0
        self.skipped       = 0
        self.failed        = 0

    def _scan(self):
        self.videos = sorted([
            f for f in os.listdir(self.input_folder)
            if f.lower().endswith(VIDEO_EXTENSIONS)
        ])

    def _process(self, filename, idx):
        input_path  = os.path.join(self.input_folder,  filename)
        output_path = os.path.join(self.output_folder, filename)

        width, height, fps = get_video_info(input_path)
        already_ok = (
            abs(fps - TARGET_FPS) < FPS_TOLERANCE
            and width  == TARGET_WIDTH
            and height == TARGET_HEIGHT
        )

        print(f"[{idx}/{len(self.videos)}] {filename}  {width}x{height} {fps:.2f}fps", end=" ... ")

        if already_ok:
            shutil.copy2(input_path, output_path)
            self.skipped += 1
            print("copied")
            return

        success = normalize_video(input_path, output_path)
        if success:
            self.converted += 1
            print("done")
        else:
            self.failed += 1
            print("FAILED")

    def run(self):
        if not os.path.exists(self.input_folder):
            print("input folder not found:", self.input_folder)
            return

        os.makedirs(self.output_folder, exist_ok=True)
        self._scan()
        print(f"found {len(self.videos)} videos  |  target: {TARGET_WIDTH}x{TARGET_HEIGHT} @ {TARGET_FPS}fps")

        for idx, filename in enumerate(self.videos, start=1):
            self._process(filename, idx)

        print(f"\ndone  |  converted: {self.converted}  skipped: {self.skipped}  failed: {self.failed}")


normalizer = VideoNormalizer(INPUT_FOLDER, OUTPUT_FOLDER)
normalizer.run()

found 1 videos  |  target: 1080x1920 @ 25fps
[1/1] normal_001.mp4  1080x1920 24.66fps ... copied

done  |  converted: 0  skipped: 1  failed: 0
